In [1]:
from langchain_groq import ChatGroq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0,
    groq_api_key=" ",
    model_name="llama-3.3-70b-versatile"
)

response = llm.invoke("The first person to land on moon was...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [6]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://jobs.nike.com/job/R-33460")
page_data = loader.load().pop().page_content
print(page_data)





















Nike Careers











































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



Select a Lang

In [7]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [8]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'role': 'Kaufmann/-frau im Einzelhandel (Ausbildung)',
  'experience': 'Entry-level',
  'skills': 'Retail, Sales',
  'description': '37.5H - Metzingen (w/m/d)'},
 {'role': 'Supervisor',
  'experience': 'Experienced',
  'skills': 'Leadership, Retail Management',
  'description': 'Nike Bend'},
 {'role': 'Nike Operational Teamleiter (Lead)',
  'experience': 'Experienced',
  'skills': 'Leadership, Operations Management',
  'description': '38.5 (h) – NFS Parndorf'},
 {'role': 'Retail Assistant (Athlete)',
  'experience': 'Entry-level',
  'skills': 'Retail, Sales',
  'description': 'P/T 20H TEMP- Cheshire Oaks'},
 {'role': 'Athlete III',
  'experience': 'Entry-level',
  'skills': 'Retail, Sales',
  'description': 'Hefei, Anhui, China'},
 {'role': 'Team Leader Nike (Responsable d’équipe)',
  'experience': 'Experienced',
  'skills': 'Leadership, Retail Management',
  'description': 'H / F CDI - Mulhouse'},
 {'role': 'Retail Associate',
  'experience': 'Entry-level',
  'skills': 'Retail, Sale

In [9]:
type(json_res)

list

In [11]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [12]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])


In [15]:
links = collection.query(query_texts=["Experience in Python", "Expertise in React Native"], n_results=2).get('metadatas', [])
linksjob = json_res
job['skills']


[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/react-native-portfolio'},
  {'links': 'https://example.com/react-portfolio'}]]

In [20]:
job = json_res
job[0]

{'role': 'Kaufmann/-frau im Einzelhandel (Ausbildung)',
 'experience': 'Entry-level',
 'skills': 'Retail, Sales',
 'description': '37.5H - Metzingen (w/m/d)'}

In [21]:
job = json_res[0]

In [22]:
job['skills']

'Retail, Sales'

In [23]:
job = json_res[0]
job['skills']

'Retail, Sales'

In [24]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Enhancing Retail Operations with AtliQ's Automated Solutions

Dear Hiring Manager,

I came across the job posting for Kaufmann/-frau im Einzelhandel (Ausbildung) at your esteemed organization, and I am excited to introduce AtliQ, an AI & Software Consulting company that can help streamline your retail operations. As a business development executive at AtliQ, I believe our expertise in automation and process optimization can be a perfect fit for your team.

With our tailored solutions, we can help enhance sales, retail, and overall customer experience. Our team has a proven track record of empowering enterprises with scalable, efficient, and cost-effective solutions. I would like to highlight a few examples from our portfolio that demonstrate our capabilities:

* Our machine learning expertise, showcased in projects like https://example.com/ml-python-portfolio, can help you analyze customer behavior, predict sales trends, and optimize inventory management.
* Our proficiency in 